# Week 10 Lecture 1
## Filtering, Transforming, and Plotting
- We will look at a number of techniques for filtering and transforming data

In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "code": [8876, 5543, 7846, 2395],
        "quantity": [1, 22, 4, 6],
        "price": [0.5, 0.25, 1.99, 0.3],
    }
)

### Filtering
- Pyret way:
```arr
fun is-high-quantity(r :: Row) -> Boolean:
  r["quantity"] >= 10
end

large-orders = filter-with(orders, is-high-quantity)
```
- pandas uses boolean based filtering to select values from a Series

In [ ]:
orders["quantity"] >= 10

- A Series of booleans can be used to select from the original DataFrame

In [ ]:
orders[orders["quantity"] >= 10]

- We can use the `.isn()` method to check membership of a list rather than write complex selection statements

In [ ]:
orders["code"].isin(["8876", "7846"])

- We can check for matching values within a range using the `between()` method

In [ ]:
orders["quantity"].between(3, 6)

## Transforming Values
Transforming existing columns (like Pyret's transform-column)

Pyret way:
```arr
fun increase-price(price :: Number) -> Number:
  price * 1.5
end

increased-price = transform-column(orders, "price", increase-price)
```

- We can increase all values inside a numerical Series by performing the calculation
- The new Series can be assigned to the column

In [ ]:
orders["price"] = orders["price"] * 1.5

- We can also create new columns by performing a calculation

In [ ]:
orders["total"] = orders["price"] * orders["quantity"]

## Transforming Values: Map
- A `map` can be used to apply a function element-by-element on all values in a Series
- We can use a dictionary for mapping values, for example mapping codes to product names

In [ ]:
products = {
    2395: "Grapes",
    5543: "Orange",
    7846: "Watermelon",
    8876: "Apple",
}

orders["product"] = orders["code"].map(products)

- We can also use a lambda/function for complex operations
- We should always prefer a [universal function](https://numpy.org/doc/stable/reference/ufuncs.html) over a for loop when working with NumPy/pandas
    - Universal functions are optimised for array based operations

In [ ]:
def uppercase(s):
    return s.upper()


orders["product"].map(uppercase)

# same as
# orders["product"].map(lambda s: s.upper())
# orders["product"].map(str.upper)

## Apply
- We can [apply a function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.apply.html#pandas.DataFrame.apply) across each row or column of a DataFrame
- It is similar to writing a for loop across an axis of a DataFrame
- Does not accept a dictionary, and is slower than a Map
- This is similar to adding new columns using Pyret's `build-column`

```arr
fun size-of-order(r :: Row) -> Number:
  if r["total"] > 5:
    "large"
  else:
    "small"
  end
end

order-size = build-column(orders, "size", size-of-order)
```
- In pandas, a lambda is applied row-wise by setting the argument `axis=1`
  - By default, apply works column-wise
- Note the use of an inline if statement with the lambda

In [ ]:
orders["size"] = orders.apply(lambda f: "large" if f["total"] > 5 else "small", axis=1)

## Sorting
The `sort_values()` method can be used to [sort a DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html) by a particular column


Pyret way:
```arr
order-by(orders, "quantity", true)   # ascending
order-by(orders, "quantity", false)  # descending
```

Pandas:

In [ ]:
orders.sort_values('quantity')                    # ascending

In [ ]:
orders.sort_values('quantity', ascending=False)   # descending

- We can also sort using multiple columns using a list

In [ ]:
orders.sort_values(["price", "quantity"])

## Visualisation
- We can create simple visualisations using the [plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html) method on a DataFrame
- This give us a simpler interface for using the [matplotlib](https://matplotlib.org/) library

In [ ]:
import matplotlib.pyplot as plt

- We can create a simple bar chart by using `kind=bar`

In [ ]:
orders.plot(kind="bar", x="product", y="quantity", title="orders")

### Line Plot
- We can also use methods from the matplotlib library to customise plotsm

In [ ]:
df = pd.DataFrame(
    {
        "Year": ["2020", "2021", "2022", "2023"],
        "Sales_A": [200, 520, 600, 450],
        "Sales_B": [250, 500, 550, 700],
    }
)

# set the index to 'Year'
df = df.set_index("Year")

# Line chart
df.plot(kind="line")
plt.title("Sales Over Time")
plt.xlabel("Year")
plt.ylabel("Sales")
plt.show()

### Bar Chart

In [ ]:
df.plot(kind="bar")
plt.title("Bar Plot of Sales")
plt.xlabel("Year")
plt.ylabel("Sales")
plt.show()

### Scatter Plot
Scatter plots are useful for visualising the relationship between two variables

In [ ]:
df2 = pd.DataFrame(
    {"Height": [150, 160, 170, 180, 190], "Weight": [50, 60, 65, 70, 80]}
)

df2.plot(kind="scatter", x="Height", y="Weight", color="red")
plt.title("Height vs. Weight")
plt.show()

### Histogram
Histograms allow you to visualise the distribution of a single variable.

In [ ]:
df2["Height"].plot(kind="hist")

We need more data for this chart to become useful. We can use numpy to create some random numbers for us.

In [ ]:
import numpy as np

In [ ]:
# Create 200 random numbers around the mean 175
df3 = pd.DataFrame({"height": np.random.normal(175, 10, 200)})

df3.plot(kind="hist")

### Box Plot
Box plots give insight into the distribution, outliers, and quartiles of the data

In [ ]:
df2.plot(kind="box")

## Class Exercises
- Load the `workouts.csv` into a DataFrame called `workouts`
- Load `photos.csv` into a file called `photos`
### Filtering
- Filter your `workout` DataFrame to show only workouts longer than `45` minutes.
  - Use the `sum` function to calculate the number of rows containing those workouts
- From the photos dataset, filter to show only photos with subject `"Forest"`.

### Sorting
- Sort your `workout` DataFrame by duration from longest to shortest.
- Sort the `photos` dataset by `date` (ascending), then print the first 5 rows.

### Adding new columns
- Add a column called `workout_category` that is `"long"` if duration ≥ `60`, `"medium"` if `30-59`, `"short"` otherwise.
- In the photos dataset, add a column called `is_landscape` that contains the value `True` if the location contains `"Park"` or `"Garden"`.

### Transforming columns
- Convert all workout durations from minutes to hours (divide by 60).
- In the photos dataset, transform the `subject` column to be all uppercase.

### Combining operations
- Filter `workouts` to only `"long"` category, then sort by calories burned.
- What is the average duration of workouts in each category?

### Plotting
- Create a bar chart that shows the number of pictures taken per `Subject` in the `photos` dataset
- Create a scatter plot that shows the relationship between `calories_burned` and `heart_rate`
  - Note that this dataset contains mock data!